# ML Prediction & Walk-Forward Validation

This notebook explores the **ML prediction layer** of `equity_lake`:

- **`PriceForecaster`** — XGBoost binary classifier wrapping training, inference, and walk-forward backtesting
- **`v1_direction`** mode — next-day directional prediction (up/down)
- **`v2_meta_label`** mode — triple-barrier meta-labeling with candidate trade events
- **`PurgedEmbargoedWalkForwardSplitter`** — time-series safe cross-validation with purging and embargo
- Visualization of model diagnostics, feature importance, and backtest equity curves

**Prerequisites:** Ingested price data and computed feature parquets in `data/lake/features/`.

In [1]:
import sys
sys.path.insert(0, "../../src")

from equity_lake.ml.forecasting import PriceForecaster
from equity_lake.ml import run_prediction_job
from equity_lake.ml.labeling import apply_triple_barrier_labels
from equity_lake.ml.candidates import build_candidate_frame, DEFAULT_BACKTEST_STRATEGY
from equity_lake.ml.validation import PurgedEmbargoedWalkForwardSplitter, run_purged_walk_forward_validation
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
from datetime import date, timedelta

print("ML modules loaded")

ML modules loaded


## 1. v1_direction Mode — Next-Day Direction Prediction

The **v1_direction** mode trains a binary XGBoost classifier to predict whether the next day's return is positive (`target=1`) or negative (`target=0`).

- Label: `(next_day_return > 0) -> int`
- Feature columns: all numeric columns except the `NON_FEATURE_COLUMNS` set
- Default split: 80/20 train/validation

In [2]:
ticker = "AAPL"
end_date = date.today() - timedelta(days=1)
start_date = end_date - timedelta(days=365)

try:
    forecaster_v1 = PriceForecaster(model_mode="v1_direction")
    model = forecaster_v1.train_model(
        ticker=ticker,
        start_date=start_date,
        end_date=end_date,
        max_model_age_days=0,
    )
    summary = forecaster_v1.last_training_summary()
    print(f"Model trained: {summary.get('model_file', 'N/A')}")
    print(f"Train rows: {summary.get('train_rows', 0)}")
    print(f"Validation rows: {summary.get('validation_rows', 0)}")
    forecaster_v1.close()
except Exception as e:
    print(f"v1 training error: {e}")
    print("\nEnsure feature parquets exist in data/lake/features/")
    summary = {}

2026-06-09 09:26:12 [info     ] duckdb_feature_view_ready      model_mode=v1_direction


2026-06-09 09:26:12 [debug    ] metadata_saved                 path=/Users/minghao/Desktop/personal/equity_lake/data/models/AAPL_xgboost_v1_direction_2026-06-08.training_metadata.json


2026-06-09 09:26:12 [debug    ] training_summary_saved         path=/Users/minghao/Desktop/personal/equity_lake/data/models/AAPL_xgboost_v1_direction_2026-06-08.training_summary.json


2026-06-09 09:26:12 [info     ] model_saved                    model_mode=v1_direction path=/Users/minghao/Desktop/personal/equity_lake/data/models/AAPL_xgboost_v1_direction_2026-06-08.pkl ticker=AAPL


Model trained: AAPL_xgboost_v1_direction_2026-06-08.pkl
Train rows: 28
Validation rows: 8


## 2. v2_meta_label Mode — Triple-Barrier Meta-Labeling

The **v2_meta_label** mode uses a two-step process:

1. **Candidate generation** — `build_candidate_frame()` identifies trade events from strategy rules (e.g. momentum)
2. **Triple-barrier labeling** — `apply_triple_barrier_labels()` assigns a `meta_label` (1=profit hit, 0=stop/loss) based on profit-taking, stop-loss, and vertical (time) barriers
3. The XGBoost model then learns to predict `meta_label` — i.e. whether to **execute** a candidate trade

In [3]:
try:
    forecaster_v2 = PriceForecaster(model_mode="v2_meta_label")
    df_raw = forecaster_v2.load_features(ticker, start_date, end_date)

    if not df_raw.empty:
        print(f"Loaded {len(df_raw)} rows for {ticker}")
        print(f"Columns: {list(df_raw.columns)[:15]}...")

        candidates = build_candidate_frame(df_raw, [DEFAULT_BACKTEST_STRATEGY])
        print(f"\nCandidate events: {len(candidates)}")
        if not candidates.empty:
            print(f"Actions: {candidates['candidate_action'].value_counts().to_dict()}")
    else:
        print("No feature data available")
except Exception as e:
    print(f"Candidate generation error: {e}")
    candidates = pd.DataFrame()
    df_raw = pd.DataFrame()

2026-06-09 09:26:12 [info     ] duckdb_feature_view_ready      model_mode=v2_meta_label


Loaded 36 rows for AAPL
Columns: ['ticker', 'date', 'open', 'high', 'low', 'close', 'volume', 'rsi_14', 'macd', 'macd_signal', 'macd_histogram', 'bb_upper', 'bb_middle', 'bb_lower', 'bb_width']...

Candidate events: 13
Actions: {'BUY': 12, 'SELL': 1}


In [4]:
if not df_raw.empty and not candidates.empty:
    try:
        labeled = apply_triple_barrier_labels(
            candidates,
            df_raw,
            vertical_barrier_days=5,
            pt_mult=1.5,
            sl_mult=1.0,
        )
        print(f"Labeled candidates: {len(labeled)}")
        if "barrier_outcome" in labeled.columns:
            print(f"\nBarrier outcomes:")
            print(labeled["barrier_outcome"].value_counts().to_string())
        if "meta_label" in labeled.columns:
            print(f"\nMeta-label distribution:")
            print(labeled["meta_label"].value_counts().to_string())
    except Exception as e:
        print(f"Triple-barrier labeling error: {e}")
else:
    print("No data for triple-barrier labeling")

Labeled candidates: 13

Barrier outcomes:
barrier_outcome
profit       8
stop         4
time_loss    1

Meta-label distribution:
meta_label
1    8
0    5


## 3. Walk-Forward Validation Visualization

`PurgedEmbargoedWalkForwardSplitter` produces rolling train/test splits with:
- **Purging** — removes training samples whose label horizon overlaps the test window
- **Embargo** — adds a gap after each test window to prevent information leakage

Below we visualize the splits as a Gantt chart.

In [5]:
n_samples = 1000
train_window = 252
test_window = 21
embargo_window = 1
label_horizon = 5

splitter = PurgedEmbargoedWalkForwardSplitter(
    train_window=train_window,
    test_window=test_window,
    embargo_window=embargo_window,
    label_horizon=label_horizon,
)

dummy_X = np.zeros((n_samples, 10))
splits = list(splitter.split(dummy_X))
print(f"Number of walk-forward folds: {len(splits)}")

train_bars = []
test_bars = []
for fold_idx, (train_idx, test_idx) in enumerate(splits):
    train_bars.append(dict(
        Fold=fold_idx + 1, Start=train_idx[0], End=train_idx[-1], Type="Train",
    ))
    test_bars.append(dict(
        Fold=fold_idx + 1, Start=test_idx[0], End=test_idx[-1], Type="Test",
    ))

all_bars = train_bars + test_bars
gantt_df = pd.DataFrame(all_bars)

fig = px.bar(
    gantt_df,
    x="End",
    y="Fold",
    color="Type",
    orientation="h",
    barmode="group",
    color_discrete_map={"Train": "#636EFA", "Test": "#EF553B"},
    title=f"Walk-Forward Splits — train_window={train_window}, test_window={test_window}, embargo={embargo_window}",
    labels={"End": "Sample Index (end)", "Fold": "Fold #"},
)
fig.update_layout(height=max(400, 30 * len(splits)), yaxis=dict(dtick=1))
fig.show()

Number of walk-forward folds: 3


## 4. Prediction Probability Distribution

Histogram of model prediction probabilities from a walk-forward backtest. Values near 0 or 1 indicate high confidence; values near 0.5 indicate uncertainty.

In [6]:
try:
    forecaster_bt = PriceForecaster(model_mode="v1_direction")
    bt_start = end_date - timedelta(days=730)
    bt_results = forecaster_bt.backtest(
        ticker=ticker,
        start_date=bt_start,
        end_date=end_date,
        train_window=500,
        retrain_interval=63,
    )
    forecaster_bt.close()

    if not bt_results.empty:
        fig = px.histogram(
            bt_results, x="probability", nbins=50,
            title="Prediction Probability Distribution",
            labels={"probability": "Predicted Probability (positive class)"},
            color_discrete_sequence=["#636EFA"],
        )
        fig.add_vline(x=0.5, line_dash="dash", line_color="red", annotation_text="Decision boundary (0.5)")
        fig.update_layout(height=450)
        fig.show()
        print(f"Backtest rows: {len(bt_results)}")
        print(f"Mean probability: {bt_results['probability'].mean():.4f}")
    else:
        raise ValueError("Empty backtest results")
except Exception as e:
    print(f"Backtest error: {e}")
    print("\nGenerating synthetic probability distribution for demo...")
    np.random.seed(42)
    synthetic_proba = np.clip(np.random.beta(2, 2, size=2000), 0.01, 0.99)
    fig = px.histogram(
        x=synthetic_proba, nbins=50,
        title="Prediction Probability Distribution (Synthetic Demo)",
        labels={"x": "Predicted Probability (positive class)"},
        color_discrete_sequence=["#636EFA"],
    )
    fig.add_vline(x=0.5, line_dash="dash", line_color="red", annotation_text="Decision boundary (0.5)")
    fig.update_layout(height=450)
    fig.show()

2026-06-09 09:26:13 [info     ] duckdb_feature_view_ready      model_mode=v1_direction


Backtest error: Not enough data for backtesting. Need at least 510 days

Generating synthetic probability distribution for demo...


## 5. Feature Importance

Top 20 features by XGBoost `feature_importances_` (gain-based).

In [7]:
try:
    forecaster_fi = PriceForecaster(model_mode="v1_direction")
    model_fi = forecaster_fi.train_model(
        ticker=ticker,
        start_date=start_date,
        end_date=end_date,
        max_model_age_days=0,
    )

    importances = model_fi.feature_importances_
    feature_names = model_fi.get_booster().feature_names
    if feature_names is None:
        df_raw_fi = forecaster_fi.load_features(ticker, start_date, end_date)
        from equity_lake.ml.forecasting import NON_FEATURE_COLUMNS
        feature_names = [c for c in df_raw_fi.columns if c not in NON_FEATURE_COLUMNS]

    fi_df = pd.DataFrame({"feature": feature_names, "importance": importances})
    fi_df = fi_df.sort_values("importance", ascending=False).head(20)

    fig = px.bar(
        fi_df, x="importance", y="feature", orientation="h",
        title="Top 20 Feature Importances (XGBoost Gain)",
        color="importance", color_continuous_scale="Viridis",
    )
    fig.update_layout(height=600, yaxis=dict(autorange="reversed"))
    fig.show()
    forecaster_fi.close()
except Exception as e:
    print(f"Feature importance error: {e}")
    print("\nGenerating synthetic feature importance for demo...")
    np.random.seed(42)
    synth_features = [f"feature_{i:03d}" for i in range(20)]
    synth_importance = np.sort(np.random.exponential(0.05, 20))[::-1]
    fi_df = pd.DataFrame({"feature": synth_features, "importance": synth_importance})
    fig = px.bar(
        fi_df, x="importance", y="feature", orientation="h",
        title="Top 20 Feature Importances (Synthetic Demo)",
        color="importance", color_continuous_scale="Viridis",
    )
    fig.update_layout(height=600, yaxis=dict(autorange="reversed"))
    fig.show()

2026-06-09 09:26:13 [info     ] duckdb_feature_view_ready      model_mode=v1_direction


2026-06-09 09:26:13 [debug    ] metadata_saved                 path=/Users/minghao/Desktop/personal/equity_lake/data/models/AAPL_xgboost_v1_direction_2026-06-08.training_metadata.json


2026-06-09 09:26:13 [debug    ] training_summary_saved         path=/Users/minghao/Desktop/personal/equity_lake/data/models/AAPL_xgboost_v1_direction_2026-06-08.training_summary.json


2026-06-09 09:26:13 [info     ] model_saved                    model_mode=v1_direction path=/Users/minghao/Desktop/personal/equity_lake/data/models/AAPL_xgboost_v1_direction_2026-06-08.pkl ticker=AAPL


## 6. Confusion Matrix

Heatmap of predicted vs actual labels from walk-forward backtest results.

In [8]:
try:
    if not bt_results.empty and "prediction" in bt_results.columns and "actual" in bt_results.columns:
        from sklearn.metrics import confusion_matrix

        cm = confusion_matrix(bt_results["actual"], bt_results["prediction"])
        labels = ["Down (0)", "Up (1)"]

        fig = go.Figure(data=go.Heatmap(
            z=cm, x=labels, y=labels,
            text=cm, texttemplate="%{text}", textfont={"size": 16},
            colorscale="Blues",
        ))
        fig.update_layout(
            title="Confusion Matrix — Walk-Forward Predictions",
            xaxis_title="Predicted", yaxis_title="Actual",
            height=500, width=600,
        )
        fig.show()

        accuracy = (bt_results["prediction"] == bt_results["actual"]).mean()
        print(f"Overall accuracy: {accuracy:.4f}")
    else:
        raise ValueError("No backtest results")
except Exception as e:
    print(f"Confusion matrix error: {e}")
    print("\nGenerating synthetic confusion matrix for demo...")
    np.random.seed(42)
    n = 500
    y_true = np.random.randint(0, 2, n)
    y_pred = np.where(np.random.rand(n) < 0.6, y_true, 1 - y_true)
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(y_true, y_pred)
    labels = ["Down (0)", "Up (1)"]
    fig = go.Figure(data=go.Heatmap(
        z=cm, x=labels, y=labels,
        text=cm, texttemplate="%{text}", textfont={"size": 16},
        colorscale="Blues",
    ))
    fig.update_layout(
        title="Confusion Matrix (Synthetic Demo)",
        xaxis_title="Predicted", yaxis_title="Actual",
        height=500, width=600,
    )
    fig.show()

Confusion matrix error: name 'bt_results' is not defined

Generating synthetic confusion matrix for demo...


## 7. Walk-Forward Backtest Equity Curve

Cumulative returns from the walk-forward backtest, assuming we trade based on model predictions.

In [9]:
try:
    if not bt_results.empty and "actual_return" in bt_results.columns:
        bt_results["strategy_return"] = np.where(
            bt_results["prediction"] == 1, bt_results["actual_return"], 0,
        )
        bt_results["cumulative_return"] = (1 + bt_results["strategy_return"]).cumprod()
        bt_results["buy_hold_cumulative"] = (1 + bt_results["actual_return"]).cumprod()

        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=bt_results["date"], y=bt_results["cumulative_return"],
            name="Strategy", line=dict(color="#636EFA", width=2),
        ))
        fig.add_trace(go.Scatter(
            x=bt_results["date"], y=bt_results["buy_hold_cumulative"],
            name="Buy & Hold", line=dict(color="#EF553B", width=2, dash="dash"),
        ))
        fig.update_layout(
            title=f"Walk-Forward Equity Curve — {ticker}",
            xaxis_title="Date", yaxis_title="Cumulative Return",
            height=500,
        )
        fig.show()

        total_return = bt_results["cumulative_return"].iloc[-1] - 1
        bh_return = bt_results["buy_hold_cumulative"].iloc[-1] - 1
        print(f"Strategy total return: {total_return:.2%}")
        print(f"Buy & Hold total return: {bh_return:.2%}")
    else:
        raise ValueError("No backtest results with returns")
except Exception as e:
    print(f"Equity curve error: {e}")
    print("\nGenerating synthetic equity curve for demo...")
    np.random.seed(42)
    dates = pd.date_range("2024-01-01", periods=252, freq="B")
    strategy_returns = np.random.normal(0.0003, 0.015, len(dates))
    bh_returns = np.random.normal(0.0002, 0.012, len(dates))

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=dates, y=(1 + strategy_returns).cumprod(),
        name="Strategy (Synthetic)", line=dict(color="#636EFA", width=2),
    ))
    fig.add_trace(go.Scatter(
        x=dates, y=(1 + bh_returns).cumprod(),
        name="Buy & Hold (Synthetic)", line=dict(color="#EF553B", width=2, dash="dash"),
    ))
    fig.update_layout(
        title="Walk-Forward Equity Curve (Synthetic Demo)",
        xaxis_title="Date", yaxis_title="Cumulative Return",
        height=500,
    )
    fig.show()

Equity curve error: name 'bt_results' is not defined

Generating synthetic equity curve for demo...


## 8. Purged Walk-Forward Validation Metrics

Use `run_purged_walk_forward_validation()` to compute fold-by-fold accuracy, precision, and recall across the entire dataset.

In [10]:
try:
    forecaster_val = PriceForecaster(model_mode="v1_direction")
    df_val = forecaster_val.load_features(ticker, start_date, end_date)

    if not df_val.empty:
        from equity_lake.ml.forecasting import NON_FEATURE_COLUMNS
        feature_cols = [c for c in df_val.columns if c not in NON_FEATURE_COLUMNS]
        df_val["target"] = (df_val["next_day_return"] > 0).astype(int)
        df_clean = df_val.dropna(subset=["target"]).copy()
        X = df_clean[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
        y = df_clean["target"].astype(int)

        metrics = run_purged_walk_forward_validation(
            X=X, y=y,
            train_window=252, test_window=21,
            embargo_window=1, label_horizon_days=1,
        )
        print("Walk-Forward Validation Metrics:")
        for k, v in metrics.items():
            print(f"  {k}: {v}")
    else:
        raise ValueError("No feature data")
    forecaster_val.close()
except Exception as e:
    print(f"Validation metrics error: {e}")
    print("\nExpected output format:")
    print("  folds: 5")
    print("  mean_accuracy: 0.53")
    print("  std_accuracy: 0.04")
    print("  mean_precision: 0.55")
    print("  mean_recall: 0.52")

2026-06-09 09:26:13 [info     ] duckdb_feature_view_ready      model_mode=v1_direction


Walk-Forward Validation Metrics:
  folds: 0
  mean_accuracy: 0.0
  mean_precision: 0.0
  mean_recall: 0.0


## Next Steps

- **[07-signal-scanning.ipynb](07-signal-scanning.ipynb)** — Generate trade signals from features + ML predictions
- **[08-backtesting.ipynb](08-backtesting.ipynb)** — Full backtesting framework with vectorbt
- **[04-hamilton-features.ipynb](04-hamilton-features.ipynb)** — Hamilton DAG feature engineering deep dive